# Experiment 2: Analysis of Harmonized Data

In [ ]:
import os
from pathlib import Path

if Path.cwd().name != 'eeg-site-effects':
    os.chdir('../..')
print('Working directory:', Path.cwd())

In [ ]:
%matplotlib inline

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from itertools import combinations
from sklearn.preprocessing import StandardScaler

from src.harmonization.sitewise_scaler import SiteWiseStandardScaler
from combatlearn.combat import ComBat

INFO_FILE_PATH = 'data/ELM19/filtered/ELM19_info_filtered_norm.csv'
FEATURES_FILE_PATH = 'data/ELM19/filtered/ELM19_features_filtered_norm.csv'
SAVE_DIR = 'results/figures/04_paradox_analysis/exp02_data_visualization'

COVARIATES = ['age_dec', 'patient_sex']
SITE_COLUMN = 'institution_id'

In [ ]:
try:
    info_df = pd.read_csv(INFO_FILE_PATH)
    features_df = pd.read_csv(FEATURES_FILE_PATH)
    print(f"Data loaded. Info: {info_df.shape}, Features: {features_df.shape}")
except FileNotFoundError:
    print("ERROR: Data files not found. Run on server.")
    raise

In [ ]:
X = features_df
sites = info_df[SITE_COLUMN]
all_hospitals = np.unique(sites)

harmonizers = {
    "raw": None,
    "sitewise": SiteWiseStandardScaler(batch=sites),
    "combat": ComBat(batch=sites, method='johnson'),
    "neurocombat": ComBat(
        batch=sites,
        discrete_covariates=info_df[['patient_sex']],
        continuous_covariates=info_df[['age_dec']],
        method='fortin'
    ),
    "covbat": ComBat(
        batch=sites,
        discrete_covariates=info_df[['patient_sex']],
        continuous_covariates=info_df[['age_dec']],
        method='chen'
    ),
}

In [ ]:
transformed_datasets = {}

for method, harmonizer in harmonizers.items():
    print(f"Applying '{method}'...")
    if harmonizer is None:
        X_scaled = StandardScaler().fit_transform(X)
    else:
        X_scaled = StandardScaler().fit_transform(harmonizer.fit_transform(X))
    transformed_datasets[method] = pd.DataFrame(X_scaled, index=X.index, columns=X.columns)

In [ ]:
import os
os.makedirs(SAVE_DIR, exist_ok=True)

for method, harmonizer in harmonizers.items():
    if harmonizer is not None:
        fig = harmonizer.plot_transformation(X, reduction_method='tsne')
        fig.savefig(f"{SAVE_DIR}/{method}_tsne.png", dpi=150, bbox_inches='tight')
        plt.close(fig)
        print(f"Saved: {method}_tsne.png")

In [ ]:
all_cov_distances = []
all_cov_matrices = {}

for method, X_transformed in transformed_datasets.items():
    cov_matrices = {}
    for site in all_hospitals:
        site_data = X_transformed[sites == site]
        cov_matrices[site] = site_data.cov().values
    all_cov_matrices[method] = cov_matrices

    for site_a, site_b in combinations(all_hospitals, 2):
        f_distance = np.linalg.norm(cov_matrices[site_a] - cov_matrices[site_b], 'fro')
        all_cov_distances.append({
            'method': method,
            'comparison': f"{site_a}_vs_{site_b}",
            'frobenius_distance': f_distance
        })

cov_dist_df = pd.DataFrame(all_cov_distances)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sns.boxplot(data=cov_dist_df, x='method', y='frobenius_distance', notch=True,
            order=['raw', 'sitewise', 'combat', 'neurocombat', 'covbat'], ax=ax)
ax.set_title("Pairwise Frobenius Distance Between Hospital Covariance Matrices", fontsize=15)
ax.set_xlabel('Harmonization Method')
ax.set_ylabel('Frobenius Distance')
ax.grid(axis='y', linestyle='--', alpha=0.7)
fig.tight_layout()
fig.savefig(f"{SAVE_DIR}/frobenius_distance.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
for hospital_to_compare in range(len(all_hospitals)):
    hospital_id = all_hospitals[hospital_to_compare]
    harm_types = list(transformed_datasets.keys())

    all_vals = np.concatenate([
        all_cov_matrices[h][hospital_id].flatten() for h in harm_types
    ])
    vmax = np.percentile(np.abs(all_vals), 99)
    vmin = -vmax

    fig, axes = plt.subplots(len(harm_types), 1,
                             figsize=(8, len(harm_types) * 8),
                             sharex=True, sharey=True, dpi=150)

    for i, harm_type in enumerate(harm_types):
        axes[i].imshow(all_cov_matrices[harm_type][hospital_id], cmap='vlag', vmin=vmin, vmax=vmax)
        axes[i].set_ylabel(harm_type, fontsize=11)
        axes[i].set_xticks([])
        axes[i].set_yticks([])

    axes[0].set_title(hospital_id, fontsize=12, fontweight='bold')
    cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
    fig.colorbar(plt.cm.ScalarMappable(cmap='vlag', norm=plt.Normalize(vmin=vmin, vmax=vmax)), cax=cbar_ax)
    fig.suptitle(f"Covariance Matrices — {hospital_id}", fontsize=14, fontweight='bold')
    fig.subplots_adjust(left=0.05, right=0.9, top=0.95, bottom=0.05, hspace=0.1)

    save_path = f"{SAVE_DIR}/covariance_matrix_{hospital_id}.png"
    fig.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f"Saved: covariance_matrix_{hospital_id}.png")